In [1]:
import pandas as pd

df = pd.read_csv("books_scraped_detailed_v2.csv")

In [2]:
print(df.shape)
print(df.columns.tolist())

(60, 11)
['title', 'price', 'rating', 'availability', 'product_url', 'category', 'UPC', 'tax', 'reviews', 'quantity', 'description']


In [3]:
df.head()

,title,price,rating,availability,product_url,category,UPC,tax,reviews,quantity,description
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...,Poetry,a897fe39b1053632,0.0,0,22,It's hard to imagine a world without A Light i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,90fa61229261140a,0.0,0,20,"""Erotic and absorbing...Written with starling ..."
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...,Fiction,6957f44c3847a760,0.0,0,20,"Dans une France assez proche de la nôtre, un h..."
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...,Mystery,e00eb4fd7b871a48,0.0,0,20,"WICKED above her hipbone, GIRL across her hear..."
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...,History,4165285e1663650f,0.0,0,20,From a renowned historian comes a groundbreaki...


In [4]:
df[["title", "category", "description"]].head()

,title,category,description
0,A Light in the Attic,Poetry,It's hard to imagine a world without A Light i...
1,Tipping the Velvet,Historical Fiction,"""Erotic and absorbing...Written with starling ..."
2,Soumission,Fiction,"Dans une France assez proche de la nôtre, un h..."
3,Sharp Objects,Mystery,"WICKED above her hipbone, GIRL across her hear..."
4,Sapiens: A Brief History of Humankind,History,From a renowned historian comes a groundbreaki...


###### Check for missing text

In [5]:
print("Missing titles:", df["title"].isnull().sum())
print("Missing categories:", df["category"].isnull().sum())
print("Missing descriptions:", df["description"].isnull().sum())

Missing titles: 0
Missing categories: 0
Missing descriptions: 0


##### If descriptions have blanks, also run:

In [6]:
print("Empty descriptions:", (df["description"].fillna("") == "").sum())

Empty descriptions: 0


In [7]:
df["combined_text"] = (
    df["title"].fillna("") + " " +
    df["category"].fillna("") + " " +
    df["description"].fillna("")
)

In [8]:
df[["title", "combined_text"]].head()

,title,combined_text
0,A Light in the Attic,A Light in the Attic Poetry It's hard to imagi...
1,Tipping the Velvet,"Tipping the Velvet Historical Fiction ""Erotic ..."
2,Soumission,Soumission Fiction Dans une France assez proch...
3,Sharp Objects,Sharp Objects Mystery WICKED above her hipbone...
4,Sapiens: A Brief History of Humankind,Sapiens: A Brief History of Humankind History ...


In [9]:
print(df["combined_text"].iloc[0])

A Light in the Attic Poetry It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for 

##### Import TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

##### Create the TF-IDF matrix

In [12]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["combined_text"])

##### Check the Matrix

In [13]:
print(tfidf_matrix.shape)

(60, 3349)


It means 60 books, one row for each book. Thousands of word features, each column represents a term.

##### Calculate cosine similarity

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print(similarity_matrix.shape)

(60, 60)


In [17]:
print(similarity_matrix[0])

[1.         0.         0.         0.01194365 0.00194998 0.00208988
 0.0108969  0.00971502 0.01693477 0.0356985  0.         0.05714258
 0.00348272 0.         0.00713588 0.02764266 0.03633444 0.00800534
 0.         0.00498767 0.         0.00435191 0.         0.00228937
 0.00557155 0.02894883 0.00727417 0.03603066 0.01628089 0.00464652
 0.00594041 0.04131347 0.00272954 0.00250808 0.02305194 0.
 0.0197305  0.         0.00891822 0.06014745 0.05058807 0.02519307
 0.         0.00994013 0.00783764 0.05929891 0.01072362 0.02257052
 0.0124221  0.01046797 0.00706729 0.0102537  0.0035082  0.00287457
 0.00753686 0.00628353 0.00205806 0.02053357 0.         0.00550886]


there will be 60 similarity scores. The first score should be approximately 1.0 because first book is used to compare with itself.

In [18]:
similarity_scores = list(enumerate(similarity_matrix[0]))

sorted_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

sorted_scores[1:6]

[(39, 0.060147448907194895),
 (45, 0.0592989130792024),
 (11, 0.05714258117536029),
 (40, 0.05058806521952742),
 (31, 0.04131346832151001)]

this returns the top five most similar books as pairs of (book_index, similarty _score). Once we get that output, we'll turn those indices into actual bok titles and build our recommendation function.

0.06 shows relatively weak overlap. 

Why our scores might be low? Our dataset only contains only 60 bookds across many different genres. TF-IDF also relies on shared words rather than understanding that different words can have similar meanings.
For example, "a magical kingdom" and "a fantasy realm" are semantically similar, but TF-IDF may give them limited similarity because the words differ.

In [20]:
for index, score in sorted_scores[1:6]:
    print(
        df.iloc[index]["title"],
        "| Category:", df.iloc[index]["category"],
        "| Similarity:", round(score, 4)
    )

You can't bury them all: Poems | Category: Poetry | Similarity: 0.0601
When We Collided | Category: Contemporary | Similarity: 0.0593
Shakespeare's Sonnets | Category: Poetry | Similarity: 0.0571
Slow States of Collapse: Poems | Category: Poetry | Similarity: 0.0506
The Five Love Languages: How to Express Heartfelt Commitment to Your Mate | Category: Nonfiction | Similarity: 0.0413


These results are quite encouraging. Our recommender is identifying a meaningful pattern: 3 of the 5 are poetry books, which makes sense for A Light in the Attic. The model is finding relevant books despite having only 60 books and a relatively small amount of text.

#### Next: Build a reusable recommendation function

In [21]:
def recommend_books(title, df, similarity_matrix, top_n=5):

    # Find the book's index
    matches = df[
        df["title"].str.lower() == title.lower()
    ]

    if matches.empty:
        print("Book not found.")
        return None

    book_index = matches.index[0]

    # Get similarity scores
    scores = list(enumerate(similarity_matrix[book_index]))

    # Sort from highest to lowest
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude the book itself
    scores = [
        (index, score)
        for index, score in scores
        if index != book_index
    ]

    # Select top N
    top_scores = scores[:top_n]

    # Build recommendation results
    recommendations = []

    for index, score in top_scores:
        recommendations.append({
            "title": df.iloc[index]["title"],
            "category": df.iloc[index]["category"],
            "similarity": round(score, 4)
        })

    return pd.DataFrame(recommendations)

In [22]:
recommend_books(
    "A Light in the Attic",
    df,
    similarity_matrix
)

,title,category,similarity
0,You can't bury them all: Poems,Poetry,0.0601
1,When We Collided,Contemporary,0.0593
2,Shakespeare's Sonnets,Poetry,0.0571
3,Slow States of Collapse: Poems,Poetry,0.0506
4,The Five Love Languages: How to Express Heartf...,Nonfiction,0.0413


In [23]:
df["title"].head(10)

0                                 A Light in the Attic
1                                   Tipping the Velvet
2                                           Soumission
3                                        Sharp Objects
4                Sapiens: A Brief History of Humankind
5                                      The Requiem Red
6    The Dirty Little Secrets of Getting Your Dream...
7    The Coming Woman: A Novel Based on the Life of...
8    The Boys in the Boat: Nine Americans and Their...
9                                      The Black Maria
Name: title, dtype: object

In [24]:
recommend_books(
    "Sapiens: A Brief History of Humankind",
    df,
    similarity_matrix
)

,title,category,similarity
0,"Unbound: How Eight Technologies Made Us Human,...",History,0.1096
1,You can't bury them all: Poems,Poetry,0.0525
2,Worlds Elsewhere: Journeys Around Shakespeare’...,Nonfiction,0.0517
3,Mesaerion: The Best Science Fiction Stories 18...,Science Fiction,0.0424
4,How Music Works,Music,0.0406


The recommender works for other titles. The next useful improvement is to boost category importance. Right now, title, category, and description are all just concatenated once. We can make category count more strongly like this:

In [25]:
df["combined_text"] = (
    df["title"].fillna("") + " " +
    (df["category"].fillna("") + " ") * 3 +
    df["description"].fillna("")
)

In [26]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    df["combined_text"]
)

similarity_matrix = cosine_similarity(
    tfidf_matrix
)

In [27]:
recommend_books(
    "Sapiens: A Brief History of Humankind",
    df,
    similarity_matrix
)

,title,category,similarity
0,"Unbound: How Eight Technologies Made Us Human,...",History,0.1346
1,Worlds Elsewhere: Journeys Around Shakespeare’...,Nonfiction,0.0556
2,You can't bury them all: Poems,Poetry,0.0532
3,Mesaerion: The Best Science Fiction Stories 18...,Science Fiction,0.0442
4,The Five Love Languages: How to Express Heartf...,Nonfiction,0.0388
